# ORATS SOXX 单日 25Δ、30日隐含波动率偏斜公式预检

只用已归档的2024 SOXX源审计行，线性Delta与期限插值；没有模型、收益或账户。

In [1]:
import sys,json,hashlib,math
from pathlib import Path
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
card=root/'research/stage1_orats_soxx_25delta_constructibility_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['source'];assert plan['frozen'] and not (card/'run_state.json').exists();sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
parent=root/src['card'];a=parent/src['audit_artifact'];assert sha(a)==src['audit_sha256']
receipt=json.loads((parent/'nas_source_archive.json').read_text());assert receipt['remote_and_scoped_restore_verified'] and receipt['sha256']==src['NAS_archive_sha256'] and receipt['members'][str(a.relative_to(root))]==src['audit_sha256']
state={'plan_sha256':sha(card/'plan.json'),'phase':'audit_read_attempted','external_GETs':0,'raw_ZIP_restores':0,'SOXX_future_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
audit=json.loads(a.read_text());assert audit['sample_date']=='2024-01-03' and audit['SOXX_strike_rows']==1124
formula=plan['formula'];target_days=formula['tenor_calendar_days'];below=[e for e in audit['SOXX_expirations'] if e['calendar_days']<target_days];above=[e for e in audit['SOXX_expirations'] if e['calendar_days']>target_days]
assert below and above;lo=max(below,key=lambda e:e['calendar_days']);hi=min(above,key=lambda e:e['calendar_days']);assert [lo['calendar_days'],hi['calendar_days']]==[16,44]
def iv_at_delta(exp,side,target):
 q=exp[side];l=q['lower'];u=q['upper']
 stem='call' if side=='call25' else 'put';delta_key=stem+'_delta';iv_key=stem+'_mid_iv';bid_key=stem+'_bid';ask_key=stem+'_ask'
 assert l and u and l[delta_key]<=target<=u[delta_key] and l[delta_key]<u[delta_key]
 assert all(math.isfinite(float(v)) for row in (l,u) for v in (row[iv_key],row[bid_key],row[ask_key]))
 assert all(row[bid_key]>0 and row[ask_key]>=row[bid_key] and row[iv_key]>0 for row in (l,u))
 weight=(target-l[delta_key])/(u[delta_key]-l[delta_key]);return float(l[iv_key]+weight*(u[iv_key]-l[iv_key]))
call16=iv_at_delta(lo,'call25',formula['call_delta_target']);call44=iv_at_delta(hi,'call25',formula['call_delta_target']);put16=iv_at_delta(lo,'put_minus25',formula['put_delta_target']);put44=iv_at_delta(hi,'put_minus25',formula['put_delta_target'])
w=(target_days-lo['calendar_days'])/(hi['calendar_days']-lo['calendar_days']);call30=call16+w*(call44-call16);put30=put16+w*(put44-put16);skew=put30-call30
result={'decision':'single_day_SOXX_25delta_30d_skew_constructible_source_only','plan_sha256':sha(card/'plan.json'),'source_audit_sha256':src['audit_sha256'],'sample_date':audit['sample_date'],'tenor_brackets_days':[lo['calendar_days'],hi['calendar_days']],'tenor_interpolation_weight':w,'call_25delta_mid_IV_by_tenor':{'16d':call16,'44d':call44},'put_minus25delta_mid_IV_by_tenor':{'16d':put16,'44d':put44},'call_25delta_30d_mid_IV':call30,'put_minus25delta_30d_mid_IV':put30,'put_minus_call_30d_IV_decimal':skew,'put_minus_call_30d_IV_vol_percentage_points':100*skew,'source_2011_coverage_qualified':False,'historical_first_delivery_qualified':False,'market_outcomes_read':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_NAS_sha256':src['NAS_archive_sha256'],'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['One 2024 sample calculation is not historical factor coverage, predictive skill or a Stage1 fee-net account.','ORATS CSV has date but not historical first delivery timestamps.','No 2011-specific chain or full-calendar missingness demonstrated.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['phase']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print(result)

{'decision': 'single_day_SOXX_25delta_30d_skew_constructible_source_only', 'plan_sha256': '832345da475766e1b4924685505656a86d41eaf057003528561236e145a4521a', 'source_audit_sha256': 'ff8e4d6aa17774daa88659150aca5f7b4f1f1a01504cb2dc7ae7fe185a090a3b', 'sample_date': '2024-01-03', 'tenor_brackets_days': [16, 44], 'tenor_interpolation_weight': 0.5, 'call_25delta_mid_IV_by_tenor': {'16d': 0.2266748994702137, '44d': 0.2348154591152815}, 'put_minus25delta_mid_IV_by_tenor': {'16d': 0.2509287787266665, '44d': 0.26713634312627627}, 'call_25delta_30d_mid_IV': 0.2307451792927476, 'put_minus25delta_30d_mid_IV': 0.2590325609264714, 'put_minus_call_30d_IV_decimal': 0.028287381633723824, 'put_minus_call_30d_IV_vol_percentage_points': 2.8287381633723823, 'source_2011_coverage_qualified': False, 'historical_first_delivery_qualified': False, 'market_outcomes_read': 0, 'model_fits': 0, 'native_accounts': 0}


# ORATS SOXX 公开样本：25Δ、30日偏斜可按冻结公式构造

只用已归档的 2024-01-03 SOXX 来源审计行：分别在16日、44日到期链内，以有效双边报价和正值mid IV的两档Delta线性插到 +0.25 看涨、−0.25 看跌，再按期限线性插到30日。ORATS 的 `delta` 列是看涨Delta，看跌Delta按[发行人说明](https://orats.com/near-eod-data)为其减1。没有恢复66.8MB原件、额外网页请求或读取 SOXX 未来收益。

单日示例：30日看涨25Δ mid IV **0.230745**，看跌−25Δ mid IV **0.259033**，看跌减看涨偏斜 **0.028287**，即 **2.829个波动率百分点**。16日与44日各自的源IV和插值、期限权重0.5见 `result.json`。这是字段及公式的可构造性核验，**不是预测效果**，也没有模型拟合或账户。

2011–2022 SOXX 全期行权价链、原始报价时刻、夜间首次送达与修订版本仍未证明。不能用这个2024单点作为训练集、独立验证或 Stage1 0bp 超额。已更新的ORATS只询样草稿专门索取2011/2012原始 SOXX 样本；发送仍需用户授权，不涉及购买。
